# 01. Εξερευνητική λήψη εξωτερικών δεδομένων από το Renewables.ninja

Αυτό το notebook διατηρείται ως **πρώιμο exploratory / historical-context notebook** για λήψη εξωτερικών δεδομένων αιολικής παραγωγής μέσω του **Renewables.ninja API**.

Ο ρόλος του είναι περιορισμένος και συγκεκριμένος:

- να δείξει ένα καθαρό και αναπαραγώγιμο παράδειγμα API πρόσβασης,
- να χρησιμοποιήσει ασφαλή φόρτωση credentials από το τοπικό αρχείο `.env`,
- και να αποθηκεύσει ένα μικρό **exploratory raw artifact** για τοπικό έλεγχο.

Το notebook **δεν** αποτελεί μέρος του current canonical forecasting pipeline του repository, το οποίο βασίζεται στο **DaKS / Kassel dataset** και ξεκινά από το `02_kassel_exploration.ipynb`.

Επίσης, το notebook **δεν** τεκμηριώνει:

- benchmark αποτέλεσμα,
- canonical thesis dataset,
- νέο forecasting claim,
- diagnostics / PHM functionality,
- ή ολοκληρωμένη digital twin υλοποίηση.

## Ρύθμιση περιβάλλοντος, ασφάλεια και όρια του notebook

Σε αυτό το βήμα:

- φορτώνουμε τις απαραίτητες βιβλιοθήκες,
- εντοπίζουμε με ασφαλή τρόπο το root του project,
- διαβάζουμε το `NINJA_API_TOKEN` από το τοπικό `.env`,
- και προετοιμάζουμε τον φάκελο εξαγωγής του exploratory artifact.

Σημαντικές σημειώσεις:

- το API token **δεν** γράφεται μέσα στο notebook,
- το παραγόμενο CSV είναι **exploratory raw export** και όχι canonical processed artifact του κύριου pipeline,
- το notebook χρησιμοποιείται μόνο ως υποστηρικτικό external-data utility και όχι ως upstream benchmark στάδιο.

In [1]:
from __future__ import annotations

from pathlib import Path
import os

import pandas as pd
import requests
from dotenv import load_dotenv
from IPython.display import display


def εντοπισμός_root_έργου(αρχικό_path: Path) -> Path:
    """
    Εντοπίζει με ασφαλή τρόπο το root του repository.

    Η αναζήτηση γίνεται ανοδικά μέχρι να βρεθεί φάκελος που
    περιέχει βασικά στοιχεία της δομής του project.
    """
    for υποψήφιο_path in [αρχικό_path.resolve(), *αρχικό_path.resolve().parents]:
        if (υποψήφιο_path / "README.md").exists() and (υποψήφιο_path / "data").exists():
            return υποψήφιο_path

    raise FileNotFoundError(
        "Δεν ήταν δυνατός ο εντοπισμός του root του project από το τρέχον working directory."
    )


# Εντοπισμός root του project
PROJECT_ROOT = εντοπισμός_root_έργου(Path.cwd())

# Ορισμός βασικών διαδρομών
ENV_PATH = PROJECT_ROOT / ".env"
RAW_EXPORT_DIR = PROJECT_ROOT / "data" / "raw"

# Δημιουργία φακέλου εξαγωγής αν δεν υπάρχει ήδη
RAW_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Έλεγχος ύπαρξης του αρχείου .env πριν από τη φόρτωση
if not ENV_PATH.exists():
    raise FileNotFoundError(
        "Δεν βρέθηκε το αρχείο .env στο root του project. "
        "Δημιούργησέ το πριν συνεχίσεις."
    )

# Φόρτωση μεταβλητών περιβάλλοντος
load_dotenv(ENV_PATH)

# Ανάκτηση API token
NINJA_API_TOKEN = os.getenv("NINJA_API_TOKEN")

if not NINJA_API_TOKEN:
    raise RuntimeError(
        "Δεν βρέθηκε η μεταβλητή NINJA_API_TOKEN στο αρχείο .env."
    )

print("Ο φάκελος root του project εντοπίστηκε επιτυχώς.")
print("Το αρχείο .env φορτώθηκε επιτυχώς.")
print("Ο φάκελος exploratory exports είναι έτοιμος.")

Ο φάκελος root του project εντοπίστηκε επιτυχώς.
Το αρχείο .env φορτώθηκε επιτυχώς.
Ο φάκελος exploratory exports είναι έτοιμος.


## Λήψη ενός μικρού exploratory δείγματος από το Renewables.ninja

Στο παρόν notebook κρατάμε σκόπιμα ένα **μικρό και καθαρό single-site example** ώστε το `NB01` να μην παρουσιαστεί ως γενικευμένο data-ingestion στάδιο του κύριου pipeline.

Σε αυτό το βήμα θα:

- εκτελέσουμε ένα μόνο API request προς το `Renewables.ninja`,
- μετατρέψουμε το response σε `pandas.DataFrame`,
- ελέγξουμε βασικά την εγκυρότητα του αποτελέσματος,
- και αποθηκεύσουμε ένα **exploratory raw CSV** για τοπικό έλεγχο.

Η λογική αυτή είναι κατάλληλη για:
- αναπαραγώγιμο external-data example,
- ασφαλή notebook τεκμηρίωση,
- και exploratory χρήση,

αλλά **όχι** για canonical benchmark, κύριο forecasting dataset ή benchmark authority του repository.

In [2]:
# ============================================================
# Συνάρτηση λήψης exploratory δεδομένων από το Renewables.ninja
# ------------------------------------------------------------
# Σκοπός:
# - εκτέλεση ενός μικρού single-site API request,
# - βασικός έλεγχος εγκυρότητας του response,
# - μετατροπή του αποτελέσματος σε καθαρό DataFrame
#   με datetime index.
#
# Σημείωση:
# Το cell αυτό υποστηρίζει exploratory / notebook-level χρήση
# και δεν αποτελεί canonical data-ingestion στάδιο του κύριου
# forecasting pipeline.
# ============================================================

def μετατροπή_index_σε_timestamp(index_values: pd.Index) -> pd.DatetimeIndex:
    """
    Μετατρέπει τον index του API response σε DatetimeIndex.

    Πρώτα επιχειρεί explicit parsing για standard string timestamps
    της μορφής YYYY-MM-DD HH:MM:SS.
    Αν αυτό αποτύχει, δοκιμάζει fallback για epoch-like τιμές.
    """
    string_index = index_values.astype(str)

    # Κύρια διαδρομή: explicit parsing για standard timestamp strings
    parsed_index = pd.to_datetime(
        string_index,
        format="%Y-%m-%d %H:%M:%S",
        errors="coerce",
    )

    if parsed_index.notna().all():
        return parsed_index

    # Fallback: δοκιμή για epoch-like timestamps
    numeric_index = pd.to_numeric(index_values, errors="coerce")

    if numeric_index.notna().all():
        parsed_index = pd.to_datetime(
            numeric_index,
            unit="ms",
            errors="coerce",
        )

        if parsed_index.notna().all():
            return parsed_index

    invalid_count = int(parsed_index.isna().sum())
    raise ValueError(
        f"Απέτυχε η μετατροπή {invalid_count} timestamp τιμών του API response."
    )


def λήψη_δεδομένων_renewables_ninja(
    *,
    token: str,
    latitude: float,
    longitude: float,
    date_from: str,
    date_to: str,
    capacity: float,
    height: int,
    turbine: str,
    timeout: int = 60,
) -> pd.DataFrame:
    """
    Εκτελεί API request προς το Renewables.ninja και επιστρέφει
    ταξινομημένο pandas DataFrame με datetime index.
    """
    # URL του wind endpoint
    api_url = "https://www.renewables.ninja/api/data/wind"

    # Παράμετροι request για ένα μικρό exploratory παράδειγμα
    request_params = {
        "lat": latitude,
        "lon": longitude,
        "date_from": date_from,
        "date_to": date_to,
        "capacity": capacity,
        "height": height,
        "turbine": turbine,
        "format": "json",
    }

    # Authorization header με το token του χρήστη
    headers = {"Authorization": f"Token {token}"}

    # Εκτέλεση request με fail-fast error handling
    try:
        response = requests.get(
            api_url,
            params=request_params,
            headers=headers,
            timeout=timeout,
        )
        response.raise_for_status()
    except requests.RequestException as exc:
        raise RuntimeError(
            f"Αποτυχία κατά το API request προς το Renewables.ninja: {exc}"
        ) from exc

    # Μετατροπή του response σε JSON dictionary
    payload = response.json()

    # Έλεγχος ότι υπάρχει το βασικό πεδίο δεδομένων
    if "data" not in payload:
        raise KeyError("Το API response δεν περιέχει το αναμενόμενο πεδίο 'data'.")

    # Μετατροπή του payload σε DataFrame
    dataframe = pd.DataFrame.from_dict(payload["data"], orient="index")

    # Έλεγχος για κενό αποτέλεσμα
    if dataframe.empty:
        raise ValueError("Το API επέστρεψε κενό αποτέλεσμα.")

    # Ρητός μετασχηματισμός του index σε DatetimeIndex
    parsed_index = μετατροπή_index_σε_timestamp(dataframe.index)

    # Ορισμός καθαρού datetime index
    dataframe.index = parsed_index
    dataframe.index.name = "timestamp"

    return dataframe.sort_index()


# ------------------------------------------------------------
# Παράδειγμα μικρής exploratory λήψης για μία τοποθεσία
# ------------------------------------------------------------
renewables_ninja_df = λήψη_δεδομένων_renewables_ninja(
    token=NINJA_API_TOKEN,
    latitude=38.46,
    longitude=23.93,
    date_from="2024-01-01",
    date_to="2024-12-31",
    capacity=1.0,
    height=100,
    turbine="Vestas V80 2000",
)

print("Το exploratory API request ολοκληρώθηκε επιτυχώς.")
print(f"Οι γραμμές του δείγματος είναι: {len(renewables_ninja_df):,}")
print(f"Οι στήλες του δείγματος είναι: {list(renewables_ninja_df.columns)}")

display(renewables_ninja_df.head())

Το exploratory API request ολοκληρώθηκε επιτυχώς.
Οι γραμμές του δείγματος είναι: 8,784
Οι στήλες του δείγματος είναι: ['electricity']


,electricity
timestamp,
2024-01-01 00:00:00,0.094
2024-01-01 01:00:00,0.093
2024-01-01 02:00:00,0.090
2024-01-01 03:00:00,0.091
2024-01-01 04:00:00,0.090


In [3]:
# ============================================================
# Εξαγωγή exploratory raw CSV
# ------------------------------------------------------------
# Σκοπός:
# - αποθήκευση του δείγματος σε thesis-safe filename,
# - βασικός έλεγχος του export,
# - καθαρό notebook closure για το NB01.
#
# Σημείωση:
# Το παραγόμενο αρχείο είναι exploratory raw artifact και όχι
# canonical processed output του κύριου forecasting pipeline.
# ============================================================

# Metadata για καθαρό και αναγνώσιμο filename
site_tag = "evia"
year_tag = "2024"

output_filename = f"rninja_{site_tag}_{year_tag}_exploratory_raw.csv"
output_path = RAW_EXPORT_DIR / output_filename

# Βασικοί έλεγχοι πριν από την αποθήκευση
if renewables_ninja_df.empty:
    raise ValueError("Δεν είναι δυνατή η αποθήκευση: το DataFrame είναι κενό.")

if renewables_ninja_df.index.name != "timestamp":
    raise ValueError("Το DataFrame δεν έχει το αναμενόμενο index name 'timestamp'.")

if not renewables_ninja_df.index.is_monotonic_increasing:
    raise ValueError("Το DataFrame δεν είναι χρονικά ταξινομημένο.")

# Αποθήκευση CSV
renewables_ninja_df.to_csv(output_path)

# Προσπάθεια εμφάνισης σχετικής διαδρομής αντί για full local path
try:
    relative_output_path = output_path.relative_to(PROJECT_ROOT)
except ValueError:
    relative_output_path = output_path

print("Η αποθήκευση του exploratory CSV ολοκληρώθηκε επιτυχώς.")
print(f"Όνομα αρχείου: {output_filename}")
print(f"Σχετική διαδρομή: {relative_output_path}")
print(f"Συνολικές γραμμές: {len(renewables_ninja_df):,}")
print("Το NB01 ολοκληρώνεται με καθαρό exploratory export.")

Η αποθήκευση του exploratory CSV ολοκληρώθηκε επιτυχώς.
Όνομα αρχείου: rninja_evia_2024_exploratory_raw.csv
Σχετική διαδρομή: data\raw\rninja_evia_2024_exploratory_raw.csv
Συνολικές γραμμές: 8,784
Το NB01 ολοκληρώνεται με καθαρό exploratory export.


## Κλείσιμο notebook και ερμηνεία αποτελέσματος

Το παρόν notebook ολοκληρώνεται με επιτυχή λήψη και αποθήκευση ενός μικρού exploratory δείγματος από το `Renewables.ninja`.

Το παραγόμενο αρχείο:

- αποτελεί **exploratory raw artifact**,
- χρησιμοποιείται μόνο για τοπικό έλεγχο και τεκμηρίωση external-data πρόσβασης,
- και **δεν** αντιμετωπίζεται ως canonical input του κύριου forecasting pipeline.

Συνεπώς, το `NB01` λειτουργεί ως ένα περιορισμένου scope, αναπαραγώγιμο και thesis-safe notebook για external-data πρόσβαση, χωρίς να ανοίγει νέο benchmark claim ή νέο scientific claim.